In [147]:
import pandas as pd 
import glob 
import os 

## Demand data processing 

In [148]:
DEMAND_DIR = "/Users/thish/Documents/ADS1002/Datasets for temperature and energy demand modelling-20260731/ALL_DEMAND_DATA"

# filters out __MACOSX/.DS_Store/._ junk from Mac zip/unzip
csv_demand = glob.glob(os.path.join(DEMAND_DIR, "*.csv"))
csv_demand = [f for f in csv_demand if not os.path.basename(f).startswith("._")]

print(f"Found {len(csv_demand)} demand CSV files")

Found 1136 demand CSV files


In [149]:
df_demand_list = [pd.read_csv(file) for file in csv_demand]
combined_demand = pd.concat(df_demand_list, ignore_index=True)

In [150]:
combined_demand.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
0,NSW1,2000/09/01 00:30,8117.23667,36.72,TRADE
1,NSW1,2000/09/01 01:00,7799.70000,32.92,TRADE
2,NSW1,2000/09/01 01:30,7453.99000,27.84,TRADE
3,NSW1,2000/09/01 02:00,7095.55333,30.62,TRADE
4,NSW1,2000/09/01 02:30,6786.22167,33.53,TRADE


In [151]:
combined_demand["SETTLEMENTDATE"] = pd.to_datetime(
    combined_demand["SETTLEMENTDATE"], format="mixed", errors="coerce"
)

print("Unparseable dates:", combined_demand["SETTLEMENTDATE"].isna().sum())

Unparseable dates: 0


In [152]:
region_to_state = {
    "NSW1": "NSW",
    "VIC1": "VIC",
    "QLD1": "QLD",
    "SA1": "SA",
    "TAS1": "TAS",
}
combined_demand["state"] = combined_demand["REGION"].map(region_to_state)

print("Unmapped regions:", combined_demand["state"].isna().sum())

Unmapped regions: 0


In [153]:
print("Combined shape:", combined_demand.shape)
print("\nRows per region:")
print(combined_demand["REGION"].value_counts())
print("\nDate range:", combined_demand["SETTLEMENTDATE"].min(), "to", combined_demand["SETTLEMENTDATE"].max())
print("\nMissing values per column:")
print(combined_demand.isna().sum())

combined_demand.head()

Combined shape: (1658965, 6)

Rows per region:
REGION
NSW1    350632
VIC1    350632
SA1     350632
QLD1    350632
TAS1    256437
Name: count, dtype: int64

Date range: 2000-01-01 00:30:00 to 2020-01-01 00:00:00

Missing values per column:
REGION            0
SETTLEMENTDATE    0
TOTALDEMAND       0
RRP               0
PERIODTYPE        0
state             0
dtype: int64


,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,state
0,NSW1,2000-09-01 00:30:00,8117.23667,36.72,TRADE,NSW
1,NSW1,2000-09-01 01:00:00,7799.70000,32.92,TRADE,NSW
2,NSW1,2000-09-01 01:30:00,7453.99000,27.84,TRADE,NSW
3,NSW1,2000-09-01 02:00:00,7095.55333,30.62,TRADE,NSW
4,NSW1,2000-09-01 02:30:00,6786.22167,33.53,TRADE,NSW


## Temperature data processing

In [154]:
TEMP_DIR = "/Users/thish/Documents/ADS1002/Datasets for temperature and energy demand modelling-20260731/Temperature Data"
STN_DETAILS_PATH = "/Users/thish/Documents/ADS1002/Datasets for temperature and energy demand modelling-20260731/HM01X_StnDet_999999999743964.txt"

txt_temperature = glob.glob(os.path.join(TEMP_DIR, "*.txt"))
txt_temperature = [f for f in txt_temperature if not os.path.basename(f).startswith("._")]

print(f"Found {len(txt_temperature)} temperature station files:")
for f in txt_temperature:
    print(" -", os.path.basename(f))

Found 6 temperature station files:
 - HM01X_Data_094029_999999999743964.txt
 - HM01X_Data_086338_999999999743964.txt
 - HM01X_Data_023090_999999999743964.txt
 - HM01X_Data_066062_999999999743964.txt
 - HM01X_Data_086071_999999999743964.txt
 - HM01X_Data_040913_999999999743964.txt


In [155]:
# from BoM Notes file -- 34 fields, 1 header/description line before data starts
TEMP_COLUMNS = [
    "record_id", "station_number",
    "year_local", "month_local", "day_local", "hour_local", "minute_local",
    "year_std", "month_std", "day_std", "hour_std", "minute_std",
    "precipitation_9am_mm", "precipitation_quality",
    "air_temperature_c", "air_temperature_quality",
    "wet_bulb_temp_c", "wet_bulb_quality",
    "dew_point_temp_c", "dew_point_quality",
    "relative_humidity_pct", "relative_humidity_quality",
    "wind_speed_kmh", "wind_speed_quality",
    "wind_direction_deg", "wind_direction_quality",
    "max_gust_kmh", "max_gust_quality",
    "mslp_hpa", "mslp_quality",
    "station_level_pressure_hpa", "station_level_pressure_quality",
    "aws_flag", "end_marker",
]

In [156]:
#read and combine the temperature files 

df_temp_list = []
for file in txt_temperature:
    df = pd.read_csv(
        file, skiprows=1, header=None, names=TEMP_COLUMNS,
        na_values=["", " "], skipinitialspace=True, dtype=str,
    )
    df_temp_list.append(df)

combined_temp = pd.concat(df_temp_list, ignore_index=True)
combined_temp = combined_temp.drop(columns=["record_id", "end_marker"])

In [157]:
#convert to correct data types 

numeric_cols = [
    "station_number",
    "year_local", "month_local", "day_local", "hour_local", "minute_local",
    "year_std", "month_std", "day_std", "hour_std", "minute_std",
    "precipitation_9am_mm", "air_temperature_c", "wet_bulb_temp_c",
    "dew_point_temp_c", "relative_humidity_pct", "wind_speed_kmh",
    "wind_direction_deg", "max_gust_kmh", "mslp_hpa",
    "station_level_pressure_hpa", "aws_flag",
]
for col in numeric_cols:
    combined_temp[col] = pd.to_numeric(combined_temp[col], errors="coerce")

# quality flag columns (Y/N/W/S/I) stay as strings, fix "nan"-string bug
quality_cols = [c for c in combined_temp.columns if c.endswith("_quality")]
for col in quality_cols:
    combined_temp[col] = combined_temp[col].astype(str).str.strip().replace("nan", pd.NA)

In [158]:
# using LOCAL STANDARD TIME fields
combined_temp["datetime_std"] = pd.to_datetime(
    combined_temp[["year_std", "month_std", "day_std", "hour_std", "minute_std"]]
    .rename(columns={
        "year_std": "year", "month_std": "month", "day_std": "day",
        "hour_std": "hour", "minute_std": "minute",
    }),
    errors="coerce",
)

In [159]:
#column names for the station details file 

STN_COLUMNS = [
    "record_id", "station_number", "rainfall_district_code", "station_name",
    "date_opened", "date_closed", "latitude", "longitude", "location_method",
    "state", "height_station_m", "height_barometer_m", "wmo_index",
    "first_year", "last_year", "pct_complete",
    "pct_flag_Y", "pct_flag_N", "pct_flag_W", "pct_flag_S", "pct_flag_I",
    "end_marker",
]

In [160]:
#read in station details 
stn_details = pd.read_csv(
    STN_DETAILS_PATH, skiprows=5, header=None,
    names=STN_COLUMNS, skipinitialspace=True,
)
stn_details["station_name"] = stn_details["station_name"].str.strip()
stn_details["state"] = stn_details["state"].str.strip()
stn_details = stn_details.drop(columns=["record_id", "end_marker", "location_method"])

In [161]:
stn_details.head(6)

,station_number,rainfall_district_code,station_name,date_opened,date_closed,latitude,longitude,state,height_station_m,height_barometer_m,wmo_index,first_year,last_year,pct_complete,pct_flag_Y,pct_flag_N,pct_flag_W,pct_flag_S,pct_flag_I
0,86338,86,MELBOURNE (OLYMPIC PARK),05/2013,NaN,-37.8255,144.9816,VIC,7.5,7.5,95936,2013,2020,103,0,100,0,0,0
1,86071,86,MELBOURNE REGIONAL OFFICE,01/1908,01/2015,-37.8075,144.9700,VIC,31.2,32.2,94868,2000,2015,99,0,100,0,0,0
2,23090,23A,ADELAIDE (KENT TOWN),01/1977,NaN,-34.9211,138.6216,SA,48.0,51.0,94675,2000,2020,101,0,100,0,0,0
3,66062,66,SYDNEY (OBSERVATORY HILL),01/1858,NaN,-33.8607,151.2050,NSW,39.0,40.2,94768,2000,2020,99,0,100,0,0,0
4,94029,94,HOBART (ELLERSLIE ROAD),01/1882,NaN,-42.8897,147.3278,TAS,50.5,51.4,94970,2000,2020,110,0,100,0,0,0
5,40913,40,BRISBANE,12/1999,NaN,-27.4808,153.0389,QLD,8.1,8.3,94576,2000,2020,99,0,100,0,0,0


In [162]:
combined_temp = combined_temp.merge(
    stn_details[["station_number", "station_name", "state", "latitude", "longitude"]],
    on="station_number", how="left",
)

print("Unmatched rows:", combined_temp["station_name"].isna().sum())

Unmatched rows: 0


In [163]:
combined_temp.tail()

,station_number,year_local,month_local,day_local,hour_local,minute_local,year_std,month_std,day_std,hour_std,...,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std,station_name,state,latitude,longitude
1826239,40913,2020,1,20,7,0,2020,1,20,7,...,1007.9,N,1007.0,N,1.0,2020-01-20 07:00:00,BRISBANE,QLD,-27.4808,153.0389
1826240,40913,2020,1,20,7,30,2020,1,20,7,...,1008.0,N,1007.1,N,1.0,2020-01-20 07:30:00,BRISBANE,QLD,-27.4808,153.0389
1826241,40913,2020,1,20,8,0,2020,1,20,8,...,1008.2,N,1007.3,N,1.0,2020-01-20 08:00:00,BRISBANE,QLD,-27.4808,153.0389
1826242,40913,2020,1,20,8,30,2020,1,20,8,...,1008.2,N,1007.3,N,1.0,2020-01-20 08:30:00,BRISBANE,QLD,-27.4808,153.0389
1826243,40913,2020,1,20,9,0,2020,1,20,9,...,1008.1,N,1007.2,N,1.0,2020-01-20 09:00:00,BRISBANE,QLD,-27.4808,153.0389


In [164]:
# Regional Office (086071) closes 01/2015, Olympic Park (086338) opens 05/2013
# -- cutover at Olympic Park's opening, since it's BoM's official replacement site
MELBOURNE_CUTOVER = pd.Timestamp("2013-05-01")

is_vic = combined_temp["state"] == "VIC"
keep_86071 = is_vic & (combined_temp["station_number"] == 86071) & (combined_temp["datetime_std"] < MELBOURNE_CUTOVER)
keep_86338 = is_vic & (combined_temp["station_number"] == 86338) & (combined_temp["datetime_std"] >= MELBOURNE_CUTOVER)
keep_non_vic = ~is_vic

combined_temp = combined_temp[keep_86071 | keep_86338 | keep_non_vic].copy()

print("Duplicate state/timestamp rows remaining:", combined_temp.duplicated(subset=["state", "datetime_std"]).sum())

Duplicate state/timestamp rows remaining: 0


In [134]:
print("Combined shape:", combined_temp.shape)
print("\nRows per state:")
print(combined_temp["state"].value_counts())
print("\nAny unparseable datetimes?", combined_temp["datetime_std"].isna().sum())
print("\nMissing values per column:")
print(combined_temp.isna().sum())

combined_temp.head()

Combined shape: (1796707, 37)

Rows per state:
state
TAS    386362
SA     355906
VIC    352517
QLD    350977
NSW    350945
Name: count, dtype: int64

Any unparseable datetimes? 0

Missing values per column:
station_number                         0
year_local                             0
month_local                            0
day_local                              0
hour_local                             0
minute_local                           0
year_std                               0
month_std                              0
day_std                                0
hour_std                               0
minute_std                             0
precipitation_9am_mm               63715
precipitation_quality              63715
air_temperature_c                   1215
air_temperature_quality             1215
wet_bulb_temp_c                     4275
wet_bulb_quality                    4275
dew_point_temp_c                    1503
dew_point_quality                   1503
relative_humid

,station_number,year_local,month_local,day_local,hour_local,minute_local,year_std,month_std,day_std,hour_std,...,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std,station_name,state,latitude,longitude
0,94029,2000,1,1,2,0,2000,1,1,1,...,1019.3,N,1013.0,N,NaN,2000-01-01 01:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
1,94029,2000,1,1,2,30,2000,1,1,1,...,1019.1,N,1012.8,N,NaN,2000-01-01 01:30:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
2,94029,2000,1,1,3,0,2000,1,1,2,...,1018.9,N,1012.6,N,NaN,2000-01-01 02:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
3,94029,2000,1,1,3,30,2000,1,1,2,...,1018.7,N,1012.4,N,NaN,2000-01-01 02:30:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
4,94029,2000,1,1,4,0,2000,1,1,3,...,1018.5,N,1012.2,N,NaN,2000-01-01 03:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278


In [135]:
combined_demand.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,state
0,NSW1,2000-09-01 00:30:00,8117.23667,36.72,TRADE,NSW
1,NSW1,2000-09-01 01:00:00,7799.70000,32.92,TRADE,NSW
2,NSW1,2000-09-01 01:30:00,7453.99000,27.84,TRADE,NSW
3,NSW1,2000-09-01 02:00:00,7095.55333,30.62,TRADE,NSW
4,NSW1,2000-09-01 02:30:00,6786.22167,33.53,TRADE,NSW


In [136]:
combined_temp.head()

,station_number,year_local,month_local,day_local,hour_local,minute_local,year_std,month_std,day_std,hour_std,...,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std,station_name,state,latitude,longitude
0,94029,2000,1,1,2,0,2000,1,1,1,...,1019.3,N,1013.0,N,NaN,2000-01-01 01:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
1,94029,2000,1,1,2,30,2000,1,1,1,...,1019.1,N,1012.8,N,NaN,2000-01-01 01:30:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
2,94029,2000,1,1,3,0,2000,1,1,2,...,1018.9,N,1012.6,N,NaN,2000-01-01 02:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
3,94029,2000,1,1,3,30,2000,1,1,2,...,1018.7,N,1012.4,N,NaN,2000-01-01 02:30:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
4,94029,2000,1,1,4,0,2000,1,1,3,...,1018.5,N,1012.2,N,NaN,2000-01-01 03:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278


In [137]:
merged_df = pd.merge(
    combined_demand,
    combined_temp,
    left_on=["state", "SETTLEMENTDATE"],
    right_on=["state", "datetime_std"],
    how="left",   # keep every demand row, even if no matching temperature reading exists
)

print("Merged shape:", merged_df.shape)

Merged shape: (1658965, 42)


In [138]:
#double checking for errors in dataset when merging 
print("Demand rows before merge:", len(combined_demand))
print("Merged rows after merge:", len(merged_df))

print("\nRows with no matching temperature reading:")
print(merged_df["datetime_std"].isna().sum())

print("\nMissing values per column:")
print(merged_df.isna().sum())

merged_df.head()

Demand rows before merge: 1658965
Merged rows after merge: 1658965

Rows with no matching temperature reading:
7682

Missing values per column:
REGION                                 0
SETTLEMENTDATE                         0
TOTALDEMAND                            0
RRP                                    0
PERIODTYPE                             0
state                                  0
station_number                      7682
year_local                          7682
month_local                         7682
day_local                           7682
hour_local                          7682
minute_local                        7682
year_std                            7682
month_std                           7682
day_std                             7682
hour_std                            7682
minute_std                          7682
precipitation_9am_mm               18854
precipitation_quality              18854
air_temperature_c                   8809
air_temperature_quality             

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,state,station_number,year_local,month_local,day_local,...,max_gust_quality,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std,station_name,latitude,longitude
0,NSW1,2000-09-01 00:30:00,8117.23667,36.72,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.3,N,1006.5,N,1.0,2000-09-01 00:30:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
1,NSW1,2000-09-01 01:00:00,7799.70000,32.92,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.1,N,1006.3,N,1.0,2000-09-01 01:00:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
2,NSW1,2000-09-01 01:30:00,7453.99000,27.84,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.3,N,1006.5,N,1.0,2000-09-01 01:30:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
3,NSW1,2000-09-01 02:00:00,7095.55333,30.62,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.3,N,1006.5,N,1.0,2000-09-01 02:00:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
4,NSW1,2000-09-01 02:30:00,6786.22167,33.53,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.2,N,1006.4,N,1.0,2000-09-01 02:30:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205


In [139]:
merged_df.tail()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,state,station_number,year_local,month_local,day_local,...,max_gust_quality,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std,station_name,latitude,longitude
1658960,QLD1,2008-04-30 22:00:00,6017.13,37.17,TRADE,QLD,40913.0,2008.0,4.0,30.0,...,N,1023.2,N,1022.2,N,1.0,2008-04-30 22:00:00,BRISBANE,-27.4808,153.0389
1658961,QLD1,2008-04-30 22:30:00,5847.77,54.53,TRADE,QLD,40913.0,2008.0,4.0,30.0,...,N,1023.1,N,1022.1,N,1.0,2008-04-30 22:30:00,BRISBANE,-27.4808,153.0389
1658962,QLD1,2008-04-30 23:00:00,5665.46,38.39,TRADE,QLD,40913.0,2008.0,4.0,30.0,...,N,1023.0,N,1022.0,N,1.0,2008-04-30 23:00:00,BRISBANE,-27.4808,153.0389
1658963,QLD1,2008-04-30 23:30:00,5570.00,36.48,TRADE,QLD,40913.0,2008.0,4.0,30.0,...,N,1022.8,N,1021.8,N,1.0,2008-04-30 23:30:00,BRISBANE,-27.4808,153.0389
1658964,QLD1,2008-05-01 00:00:00,5382.89,27.06,TRADE,QLD,40913.0,2008.0,5.0,1.0,...,N,1022.5,N,1021.5,N,1.0,2008-05-01 00:00:00,BRISBANE,-27.4808,153.0389


In [140]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1658965 entries, 0 to 1658964
Data columns (total 42 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   REGION                          1658965 non-null  object        
 1   SETTLEMENTDATE                  1658965 non-null  datetime64[ns]
 2   TOTALDEMAND                     1658965 non-null  float64       
 3   RRP                             1658965 non-null  float64       
 4   PERIODTYPE                      1658965 non-null  object        
 5   state                           1658965 non-null  object        
 6   station_number                  1651283 non-null  float64       
 7   year_local                      1651283 non-null  float64       
 8   month_local                     1651283 non-null  float64       
 9   day_local                       1651283 non-null  float64       
 10  hour_local                      1651283 no

In [141]:
mismatches = merged_df[merged_df["SETTLEMENTDATE"] != merged_df["datetime_std"]]

print("Total mismatched rows:", len(mismatches))
print("\nOf these, how many have a completely missing datetime_std (no match at all)?")
print(mismatches["datetime_std"].isna().sum())

print("\nMismatches by state:")
print(mismatches["state"].value_counts())

print("\nDate range of mismatched rows:")
print(mismatches["SETTLEMENTDATE"].min(), "to", mismatches["SETTLEMENTDATE"].max())

Total mismatched rows: 7682

Of these, how many have a completely missing datetime_std (no match at all)?
7682

Mismatches by state:
state
VIC    2486
TAS    1471
SA     1455
QLD    1338
NSW     932
Name: count, dtype: int64

Date range of mismatched rows:
2000-01-01 08:00:00 to 2019-10-08 08:00:00


In [142]:
merged_df = merged_df.drop(columns=["datetime_std", "state"])

In [143]:
merged_df.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,station_number,year_local,month_local,day_local,hour_local,...,max_gust_kmh,max_gust_quality,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,station_name,latitude,longitude
0,NSW1,2000-09-01 00:30:00,8117.23667,36.72,TRADE,66062.0,2000.0,9.0,1.0,1.0,...,NaN,<NA>,1011.3,N,1006.5,N,1.0,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
1,NSW1,2000-09-01 01:00:00,7799.70000,32.92,TRADE,66062.0,2000.0,9.0,1.0,2.0,...,NaN,<NA>,1011.1,N,1006.3,N,1.0,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
2,NSW1,2000-09-01 01:30:00,7453.99000,27.84,TRADE,66062.0,2000.0,9.0,1.0,2.0,...,NaN,<NA>,1011.3,N,1006.5,N,1.0,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
3,NSW1,2000-09-01 02:00:00,7095.55333,30.62,TRADE,66062.0,2000.0,9.0,1.0,3.0,...,NaN,<NA>,1011.3,N,1006.5,N,1.0,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
4,NSW1,2000-09-01 02:30:00,6786.22167,33.53,TRADE,66062.0,2000.0,9.0,1.0,3.0,...,NaN,<NA>,1011.2,N,1006.4,N,1.0,SYDNEY (OBSERVATORY HILL),-33.8607,151.205


In [144]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1658965 entries, 0 to 1658964
Data columns (total 40 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   REGION                          1658965 non-null  object        
 1   SETTLEMENTDATE                  1658965 non-null  datetime64[ns]
 2   TOTALDEMAND                     1658965 non-null  float64       
 3   RRP                             1658965 non-null  float64       
 4   PERIODTYPE                      1658965 non-null  object        
 5   station_number                  1651283 non-null  float64       
 6   year_local                      1651283 non-null  float64       
 7   month_local                     1651283 non-null  float64       
 8   day_local                       1651283 non-null  float64       
 9   hour_local                      1651283 non-null  float64       
 10  minute_local                    1651283 no

In [145]:
merged_df = merged_df.dropna()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,station_number,year_local,month_local,day_local,hour_local,...,max_gust_kmh,max_gust_quality,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,station_name,latitude,longitude
2927,VIC1,2007-03-01 00:30:00,5246.72,36.81,TRADE,86071.0,2007.0,3.0,1.0,1.0,...,13.0,N,1008.5,N,1004.8,N,1.0,MELBOURNE REGIONAL OFFICE,-37.8075,144.9700
2928,VIC1,2007-03-01 01:00:00,5095.51,33.88,TRADE,86071.0,2007.0,3.0,1.0,2.0,...,9.4,N,1008.5,N,1004.8,N,1.0,MELBOURNE REGIONAL OFFICE,-37.8075,144.9700
2929,VIC1,2007-03-01 01:30:00,5485.07,38.47,TRADE,86071.0,2007.0,3.0,1.0,2.0,...,9.4,N,1008.4,N,1004.7,N,1.0,MELBOURNE REGIONAL OFFICE,-37.8075,144.9700
2930,VIC1,2007-03-01 02:00:00,5281.56,29.44,TRADE,86071.0,2007.0,3.0,1.0,3.0,...,11.2,N,1008.3,N,1004.6,N,1.0,MELBOURNE REGIONAL OFFICE,-37.8075,144.9700
2931,VIC1,2007-03-01 02:30:00,5030.82,23.11,TRADE,86071.0,2007.0,3.0,1.0,3.0,...,0.0,N,1008.3,N,1004.6,N,1.0,MELBOURNE REGIONAL OFFICE,-37.8075,144.9700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1658960,QLD1,2008-04-30 22:00:00,6017.13,37.17,TRADE,40913.0,2008.0,4.0,30.0,22.0,...,0.0,N,1023.2,N,1022.2,N,1.0,BRISBANE,-27.4808,153.0389
1658961,QLD1,2008-04-30 22:30:00,5847.77,54.53,TRADE,40913.0,2008.0,4.0,30.0,22.0,...,0.0,N,1023.1,N,1022.1,N,1.0,BRISBANE,-27.4808,153.0389
1658962,QLD1,2008-04-30 23:00:00,5665.46,38.39,TRADE,40913.0,2008.0,4.0,30.0,23.0,...,0.0,N,1023.0,N,1022.0,N,1.0,BRISBANE,-27.4808,153.0389
1658963,QLD1,2008-04-30 23:30:00,5570.00,36.48,TRADE,40913.0,2008.0,4.0,30.0,23.0,...,0.0,N,1022.8,N,1021.8,N,1.0,BRISBANE,-27.4808,153.0389


In [146]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1658965 entries, 0 to 1658964
Data columns (total 40 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   REGION                          1658965 non-null  object        
 1   SETTLEMENTDATE                  1658965 non-null  datetime64[ns]
 2   TOTALDEMAND                     1658965 non-null  float64       
 3   RRP                             1658965 non-null  float64       
 4   PERIODTYPE                      1658965 non-null  object        
 5   station_number                  1651283 non-null  float64       
 6   year_local                      1651283 non-null  float64       
 7   month_local                     1651283 non-null  float64       
 8   day_local                       1651283 non-null  float64       
 9   hour_local                      1651283 non-null  float64       
 10  minute_local                    1651283 no